In [0]:
import sys, os
sys.path.append(os.path.abspath(".."))

from spark_session import create_spark_session

spark = create_spark_session()

df_bronze = spark.table("bronze_vendas_delta")

df_bronze.show()

# Atualizando o preço do 'Tênis' para 10% a mais
spark.sql("""
    UPDATE bronze_vendas_delta
    SET valor = valor * 1.1
    WHERE produto = 'Tênis'
""")

# Apagando as linhas onde a quantidade é 0
spark.sql("""
    DELETE FROM bronze_vendas_delta
    WHERE quantidade = 0
""")

# Inserindo um novo produto na tabela
spark.sql("""
    INSERT INTO bronze_vendas_delta (id, produto, valor, quantidade, estado)
    VALUES (6, 'Cadeira Gamer', 1999.90, 2, 'SC')
""")

# Executando o OPTIMIZE para garantir que as operações DML sejam aplicadas corretamente
spark.sql("OPTIMIZE bronze_vendas_delta")

# Rodando o VACUUM para garantir que as alterações sejam refletidas
spark.sql("VACUUM bronze_vendas_delta RETAIN 0 HOURS")

# Exibindo os dados após as operações
df_bronze_updated = spark.table("bronze_vendas_delta")
df_bronze_updated.show()

print("✅ Operações Delta (INSERT, UPDATE, DELETE) realizadas com sucesso.")

